# TaneİZ — Tane sınıflandırıcı eğitimi (Colab, ücretsiz T4)

**Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin. Toplam süre varsayılan ayarlarla ~30–50 dk.

Önkoşul: Google Drive'da `MyDrive/taneiz/raw/` altında veri setleri, `ml/DATASETS.md`'deki klasör düzeninde:
```
MyDrive/taneiz/raw/WheatVision/<etiket>/*.jpg
MyDrive/taneiz/raw/GrainSet/wheat/<etiket>/*.jpg
MyDrive/taneiz/raw/Sune/<etiket>/*.jpg
```
Bölmeler (ve **test seti kilidi**) ile model çıktıları Drive'a yazılır; oturum kapansa da kaybolmaz.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/taneiz'
# Repo: GitHub'daysa URL'yi yazın; değilse ml/ klasörünü Drive'a 'taneiz/ml' olarak yükleyin.
REPO_URL = ''  # ör. 'https://github.com/<kullanici>/taneiz.git'

In [ ]:
import os, shutil
if REPO_URL:
    !git clone -q $REPO_URL /content/taneiz
    ML = '/content/taneiz/ml'
else:
    shutil.copytree(f'{DRIVE}/ml', '/content/ml', dirs_exist_ok=True)
    ML = '/content/ml'
%cd $ML
!pip -q install timm==1.0.11 onnx onnxruntime

In [ ]:
# Hız için ham veriyi Drive'dan yerel diske kopyala (Drive üzerinden okumak çok yavaş)
!mkdir -p /content/raw && rsync -a --info=progress2 "$DRIVE/raw/" /content/raw/
RAW = '/content/raw'
SPLITS = f'{DRIVE}/splits'
ART = f'{DRIVE}/artifacts'

## 1) Veriyi hazırla ve test setini kilitle
Kilit varsa bölme yeniden üretilmez. Eşlenmemiş klasör uyarısı görürseniz `label_map.yaml`'ı düzeltip `--force` ile tekrar çalıştırın (henüz değerlendirme yapmadıysanız).

In [ ]:
!python prepare_data.py --raw-dir $RAW --out "$SPLITS"

## 2) Eğit (EfficientNet-B0, sınıf ağırlıklı kayıp, augmentation)

In [ ]:
!python train.py --model efficientnet_b0 --epochs 12 --batch-size 64 --max-per-class 4000 --raw-dir $RAW --splits "$SPLITS" --out "$ART"

## 3) Kilitli test setinde değerlendir → model_card.json + confusion_matrix.png

In [ ]:
!python evaluate.py --raw-dir $RAW --splits "$SPLITS" --artifacts "$ART"

In [ ]:
from IPython.display import Image, display
display(Image(f'{ART}/confusion_matrix.png', width=600))

## 4) ONNX'e çevir ve PyTorch çıktılarıyla eşitliğini doğrula

In [ ]:
!python export_onnx.py --raw-dir $RAW --splits "$SPLITS" --artifacts "$ART"

## 5) Seed için numune fotoğrafları (kilitli test setinden kompozit)

In [ ]:
!python make_demo_samples.py --raw-dir $RAW --splits "$SPLITS" --out "$DRIVE/seed-samples"

## 6) İndir
`taneiz-artifacts.zip` içeriğini repoda şuralara açın:
- `artifacts/*` → `ml/artifacts/` (model.onnx, model_card.json, confusion_matrix.png, samples/)
- `seed-samples/*` → `apps/web/prisma/seed-samples/`

In [ ]:
!cd "$DRIVE" && rm -f /content/taneiz-artifacts.zip && zip -qr /content/taneiz-artifacts.zip artifacts seed-samples -x 'artifacts/model.pt'
from google.colab import files
files.download('/content/taneiz-artifacts.zip')